# Power Outage Duration by Cause Category

**Name(s)**: Dilraj Grewal

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [2]:
# TODO

## Step 2: Data Cleaning and Exploratory Data Analysis

In [3]:
import pandas as pd
import plotly.express as px

df = pd.read_excel("Project 04 Checkpoint 1 Outage.xlsx", skiprows=5)
df = df[df["YEAR"].notna()]

outages_per_year = df.groupby("YEAR").size().reset_index(name="Outage Count")

fig = px.line(
    outages_per_year,
    x="YEAR",
    y="Outage Count",
    markers=True,
    title="Number of Major Power Outages in the U.S. Over Time"
)

fig.show()

## Step 3: Assessment of Missingness

In [12]:
cust = df[['CUSTOMERS.AFFECTED', 'CAUSE.CATEGORY', 'YEAR']].copy()
cust['missing'] = cust['CUSTOMERS.AFFECTED'].isna()
cust = cust.dropna(subset=['CAUSE.CATEGORY', 'YEAR'])

obs = cust.groupby('CAUSE.CATEGORY')['missing'].mean()
obs_stat = obs.max() - obs.min()

stats = []

for _ in range(1000):
    shuffled = cust.assign(
        shuffled=np.random.permutation(cust['missing'])
    )
    stat = shuffled.groupby('CAUSE.CATEGORY')['shuffled'].mean().max() - \
           shuffled.groupby('CAUSE.CATEGORY')['shuffled'].mean().min()
    stats.append(stat)

pval_cause = np.mean(np.array(stats) >= obs_stat)

obs_year = cust.groupby('YEAR')['missing'].mean()
obs_stat_year = obs_year.max() - obs_year.min()

stats_year = []

for _ in range(1000):
    shuffled = cust.assign(
        shuffled=np.random.permutation(cust['missing'])
    )
    stat = shuffled.groupby('YEAR')['shuffled'].mean().max() - \
           shuffled.groupby('YEAR')['shuffled'].mean().min()
    stats_year.append(stat)

pval_year = np.mean(np.array(stats_year) >= obs_stat_year)

pval_cause, pval_year

(np.float64(0.0), np.float64(0.0))

In [13]:
miss_by_cause = cust.groupby('CAUSE.CATEGORY')['missing'].mean().reset_index()

fig = px.bar(
    miss_by_cause,
    x='CAUSE.CATEGORY',
    y='missing',
    title='Missingness Rate of Customers Affected by Cause',
    labels={
        'CAUSE.CATEGORY': 'Cause Category',
        'missing': 'Proportion Missing'
    }
)

fig.show()

The missingness in CUSTOMERS.AFFECTED is MNAR as the amount of affected customer may be dependent on the scale of the outage. Smaller outages may lead to reports of only a few customers effect, while the larger outages may cause a flow of significant reports. 


## Step 4: Hypothesis Testing

Null Hypothesis: Outages caused by severe weather have the same average duration.

Alternative Hypothesis: Outages caused by severe weather have longer average durations.

Test Statistic: Difference in mean
(Mean of outages caused by severe weather - Mean of outages caused by other causes)

Significance Level: 0.05

In [16]:
sw = df[df['CAUSE.CATEGORY'] == 'severe weather']['OUTAGE.DURATION'].dropna()
other = df[df['CAUSE.CATEGORY'] != 'severe weather']['OUTAGE.DURATION'].dropna()

obs = sw.mean() - other.mean()

combined = np.concatenate([sw, other])

stats = []

for _ in range(1000):
    perm = np.random.permutation(combined)
    new_sw = perm[:len(sw)]
    new_other = perm[len(sw):]
    stat = new_sw.mean() - new_other.mean()
    stats.append(stat)

pval = np.mean(np.array(stats) >= obs)

obs, pval

(np.float64(2537.8062533051298), np.float64(0.0))

The difference in mean outage duration: 2537 minutes

p-value: Approximately 0.0

Since the p-value is below the 0.05 signifiance level, we reject the null hypothesis. The outcomes provides strong evidence that there is outages caused by severe weather tend are typically longer in duration than outages caused by other factors.

## Step 5: Framing a Prediction Problem

We are going to use linear regression to predict the duration of the power outages. Our response variable is going to be OUTAGE.DURATION which will allows us to generalize the severity of the outage and help us predict how long the outages may last for. The goal is compute the RSME since it measures the mean magnitude of prediction erros in the same units as the resposne variable. Features such as CAUSE.CATEGORY,YEAR and CLIMATE.CATEGORY are going to be used for the prediction as they are most likely to be available at the time of an outage.

## Step 6: Baseline Model

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error

base = df[['OUTAGE.DURATION', 'CAUSE.CATEGORY', 'YEAR']].dropna()

X = base[['CAUSE.CATEGORY', 'YEAR']]
y = base['OUTAGE.DURATION']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['CAUSE.CATEGORY']),
    ('num', 'passthrough', ['YEAR'])
])

pipe = Pipeline([
    ('pre', pre),
    ('model', LinearRegression())
])

pipe.fit(X_train, y_train)

train_preds = pipe.predict(X_train)
test_preds = pipe.predict(X_test)

test_rmse = root_mean_squared_error(y_test, test_preds)
train_rmse = root_mean_squared_error(y_train, train_preds)

train_rmse, test_rmse

(np.float64(4887.968611172868), np.float64(7161.599577697236))

Our baseline model outputs a 4887.97 for the training RMSE and 7161.60 for the test RMSE.
Since the model only uses two features and the test RMSE is still higher than we would like, we could try adding more relevant features to reduce that number.

WEBSITE: The baseline model predicts OUTAGE.DURATION using features like CAUSE.CATEGORY and YEAR. Since CAUSE.CATEGORY is a catigorical feature, we use one-hot encoding to extract meaningful value for our model. My project utilizes a sklearn pipline to perform linear regression and analyize performance through RMSE scores on the test set.


## Step 7: Final Model

In [23]:
final = df[['OUTAGE.DURATION', 'CAUSE.CATEGORY', 'YEAR', 'CLIMATE.CATEGORY', 'CUSTOMERS.AFFECTED', 'TOTAL.SALES']].dropna()

X_final = final[['CAUSE.CATEGORY', 'YEAR', 'CLIMATE.CATEGORY', 'CUSTOMERS.AFFECTED', 'TOTAL.SALES']]
y_final = final['OUTAGE.DURATION']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    random_state=42
)

pre_f = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['CAUSE.CATEGORY', 'CLIMATE.CATEGORY']),
    ('num', StandardScaler(), ['YEAR', 'CUSTOMERS.AFFECTED', 'TOTAL.SALES'])
])

pipe_f = Pipeline([
    ('pre', pre_f),
    ('model', RandomForestRegressor(random_state=42))
])

params = {
    'model__max_depth': [5, 10, 15, None],
    'model__n_estimators': [50, 100, 200]
}

grid = GridSearchCV(
    pipe_f,
    params,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

grid.fit(X_train_f, y_train_f)

best_model = grid.best_estimator_

train_preds_f = best_model.predict(X_train_f)
test_preds_f = best_model.predict(X_test_f)

train_rmse_f = mean_squared_error(y_train_f, train_preds_f) ** 0.5
test_rmse_f = mean_squared_error(y_test_f, test_preds_f) ** 0.5

grid.best_params_, train_rmse_f, test_rmse_f

({'model__max_depth': 5, 'model__n_estimators': 100},
 np.float64(3179.62246539919),
 np.float64(3240.7860198089056))

The final model achieves a training RMSE of 3179.62 and test RMSE of 3240.79.

The new model uses grid search to find the best hyperparameters, which are 5 for max_depth and 100 for n_estimators. Furthermore, the added features(CLIMATE.CATEGORY, CUSTOMER.AFFECTED, and TOTAL.SALES) allow the model to be more flexible, hence the lower test RMSE. Since both of the output are closer than before, the model is most likely not overfitting.

## Step 8: Fairness Analysis

The fairness analysis will consist of two groups. Group X will be outages caused by severe weather and Group Y will be outages caused by other factors. We are using the RMSE as a metric for this regression problem.

Null Hypothesis: The final model treats all causes of the outages the same and any difference is by random chance

Alternative Hypothesis: The final model is unfair and RMSE for weather realted outages is higher than outages caused by other factors. 

Test Statistic: Difference in RMSE( Outages caused by severe weather - Outages caused by other factors)

Significance Level: 0.05


In [24]:
test = X_test_f.copy()
test['true'] = y_test_f
test['pred'] = test_preds_f
test['group'] = test['CAUSE.CATEGORY'] == 'severe weather'

sw = test[test['group']]
other = test[~test['group']]

rmse_sw = mean_squared_error(sw['true'], sw['pred']) ** 0.5
rmse_other = mean_squared_error(other['true'], other['pred']) ** 0.5

obs = rmse_sw - rmse_other

combined = test[['true', 'pred']].copy()
groups = test['group'].values

stats = []

for _ in range(1000):
    perm = np.random.permutation(groups)
    sw_idx = perm == True
    other_idx = perm == False

    rmse_sw_perm = mean_squared_error(
        combined['true'][sw_idx],
        combined['pred'][sw_idx]
    ) ** 0.5

    rmse_other_perm = mean_squared_error(
        combined['true'][other_idx],
        combined['pred'][other_idx]
    ) ** 0.5

    stats.append(rmse_sw_perm - rmse_other_perm)

pval = np.mean(np.array(stats) >= obs)

obs, pval

(np.float64(1023.7452543779614), np.float64(0.109))

Conclusion : The difference in RMSE is approximately 1023.75, which produces a p value of 0.109. Since the p-value is below our signifiance level, we fail to reject the nulll hypothesis. There is strong evidence to suggest that any observed difference is due to random chance and our model is most likely fair.